In [2]:
# 代码作用：读取LIMS数据汇总为 榨季期分蜜指标统计报表v1
# 核心逻辑：全局累计合格率（按月度、班次独立累计）
# 2026/04/02
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType, DateType,DoubleType
# 关键：明确导入Spark函数，避免和Python内置函数混淆
from pyspark.sql.functions import (
    current_timestamp, split, lit, col, row_number, sum as spark_sum, 
    round as spark_round, avg, when, to_date, count, coalesce
)
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()



# ===================== 1. 读取LIMS班报 成品糖产量 数据 =====================
lims_cpt_sql = """
SELECT 
    season,
    niandudm,
    gongsidm,
    rq,
    banbiedm,
    name,
    SUM(chanliang) AS chanliang
FROM (
    SELECT 
        CONCAT(
            CAST((CAST(niandudm AS INTEGER) - 1) % 100 AS STRING), 
            '/', 
            CAST(CAST(niandudm AS INTEGER) % 100 AS STRING)
        ) AS season,
        niandudm,
        gongsidm,
        rq,
        CASE 
            WHEN banbiedm = '01' THEN '甲班'
            WHEN banbiedm = '02' THEN '乙班'
            WHEN banbiedm = '03' THEN '丙班'
            ELSE COALESCE(banbiedm, '未知班别')
        END AS banbiedm,
        name,
        chanliang
    FROM dwr.dwr_lims_sugar_product_detai_f_1d 
    WHERE gongsidm = 'FNR' 
        -- AND name = '原糖'
        -- AND name = '金砂糖'
) t
GROUP BY season, niandudm, gongsidm, rq, banbiedm, name
ORDER BY rq DESC;
"""
lims_cpt_df = spark.sql(lims_cpt_sql)

# ===================== 2. 读取LIMS 精糖班报实绩 数据 =====================
lims_bbsj_sql = """
SELECT 
    CONCAT(SUBSTR(CAST(year - 1 AS STRING), 3, 2), '/', SUBSTR(CAST(year AS STRING), 3, 2)) AS season,
    company,
    date,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        WHEN banbiedm = '04' THEN '丁班'
        ELSE COALESCE(banbiedm, '未知班别')
    END AS banbiedm,
    hrtcl
FROM dwr.dwr_lims_sugar_refined_shift_actual_f_1d 
WHERE company = 'FNR'
"""
lims_bbsj_df = spark.sql(lims_bbsj_sql)


# ===================== 3. 读取LIMS 榨蔗班报实绩 数据 =====================
lims_zzsj_sql = """
SELECT 
season,
tb_gongsidm,
tb_rq,
CASE 
    WHEN tb_banbiedm = '01' THEN '甲班'
    WHEN tb_banbiedm = '02' THEN '乙班'
    WHEN tb_banbiedm = '03' THEN '丙班'
    ELSE COALESCE(tb_banbiedm, '未知班别')
END AS banbie_name,
sj_zhazheliang
FROM dwd.dwd_lims_batch_report_actual_data_f_1h WHERE tb_gongsidm = 'FNR';
"""
lims_zzsj_df = spark.sql(lims_zzsj_sql)



# ===================== 4. 读取LIMS样品分析 金砂糖 数据 =====================
lims_jst_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '金砂糖'
        AND test_name1 IN ('产量', '色值', '干燥失重', '糖度')
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '产量' THEN test_value END) AS golden_sugar_yield,        -- 金砂糖产量
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS golden_sugar_color_value, -- 金砂糖色值
    MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) AS golden_sugar_dry_loss, -- 金砂糖干燥失重
    MAX(CASE WHEN test_name1 = '糖度' THEN test_value END) AS golden_sugar_brix          -- 金砂糖糖度
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '产量' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '糖度' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_jst_df = spark.sql(lims_jst_sql)


# ===================== 5. 读取LIMS样品分析 原糖 数据 =====================
lims_yt_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '原糖'
        AND test_name1 IN ('色值', '干燥失重', '产量')   -- 增加产量
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS raw_sugar_color_value,   -- 原糖色值
    MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) AS raw_sugar_dry_loss,  -- 原糖干燥失重
    MAX(CASE WHEN test_name1 = '产量' THEN test_value END) AS raw_sugar_yield          -- 原糖产量
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '产量' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_yt_df = spark.sql(lims_yt_sql)


# ===================== 6. 读取LIMS 洄溶糖浆 数据 =====================
lims_hrtj_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '炼糖生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '洄溶糖浆'
        AND test_name1 IN ('锤度', '色值')
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS remelt_syrup_brix,    -- 洄溶糖浆锤度
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS remelt_syrup_color_value  -- 洄溶糖浆色值
FROM raw_data 
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_hrtj_df = spark.sql(lims_hrtj_sql)


# ===================== 7. 读取LIMS样品分析 丙膏 数据 =====================
lims_bg_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '丙膏'                    -- 物料改为丙膏
        AND test_name1 IN ('视纯度')                   -- 仅检测视纯度
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS propyl_paste_apparent_purity  -- 丙膏视纯度
FROM raw_data 
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_bg_df = spark.sql(lims_bg_sql)


# ===================== 8. 读取LIMS样品分析 丙糖糊 数据 =====================
lims_bth_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '丙糖糊'
        AND test_name1 IN ('色值', '锤度', '视纯度')          -- 增加视纯度
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS third_sugar_paste_color_value,   -- 丙糖糊色值
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS third_sugar_paste_brix,         -- 丙糖糊锤度
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS third_sugar_paste_apparent_purity  -- 丙糖糊视纯度
FROM raw_data 
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_bth_df = spark.sql(lims_bth_sql)

# ===================== 9. 读取LIMS样品分析 榨蔗桔水 数据 =====================
lims_zzjs_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '榨蔗桔水'                              -- 物料改为榨蔗桔水
        AND test_name1 IN ('重力纯度', '产量', '锤度', '视纯度')      -- 四个检测项目
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '重力纯度' THEN test_value END) AS cane_molasses_gravity_purity,   -- 榨蔗桔水重力纯度
    MAX(CASE WHEN test_name1 = '产量' THEN test_value END) AS cane_molasses_yield,                -- 榨蔗桔水产量
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS cane_molasses_brix,                -- 榨蔗桔水锤度
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS cane_molasses_apparent_purity     -- 榨蔗桔水视纯度
FROM raw_data 
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '重力纯度' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '产量' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_zzjs_df = spark.sql(lims_zzjs_sql)
# ===================== 10. 读取LIMS样品分析 合格率标准 数据 =====================
lims_hgbz_sql = """
SELECT * FROM dwd.dwd_lims_sugar_quality_standard_f_1h WHERE org_code = 'FNR'
"""
lims_hgbz_df = spark.sql(lims_hgbz_sql)

# ===================== 11. 读取LIMS样品分析 金砂糖膏 数据 =====================
lims_jstg_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        ord_id,
        test_name1,
        -- 保留原始文本值（用于备注）
        test_test_value AS test_value_raw,
        -- 仅对视纯度和锤度转换为数值
        CASE 
            WHEN test_name1 IN ('视纯度', '锤度') AND test_test_value REGEXP '^[0-9.]+$' 
                THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value_num
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_test_type = '样品分析'
        AND ord_sample_name = '甲膏'                     -- 样品改为甲膏
        AND test_name1 IN ('备注', '视纯度', '锤度')       -- 检测项目改为备注、视纯度、锤度
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    ord_id,
    MAX(CASE WHEN test_name1 = '备注' THEN test_value_raw END) AS remark,   -- 备注（文本）
    MAX(CASE WHEN test_name1 = '视纯度' THEN test_value_num END) AS purity,   -- 视纯度（数值）
    MAX(CASE WHEN test_name1 = '锤度' THEN test_value_num END) AS brix      -- 锤度（数值）
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, ord_id
HAVING 
    MAX(CASE WHEN test_name1 = '备注' THEN test_value_raw END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '视纯度' THEN test_value_num END) IS NOT NULL
    OR MAX(CASE WHEN test_name1 = '锤度' THEN test_value_num END) IS NOT NULL
ORDER BY record_date DESC;
"""
lims_jstg_df = spark.sql(lims_jstg_sql)
lims_jstg_df.show()

Spark 启动中...
+-------+---------------+-----------+----------+---------------+--------------+------+------+-----+
|company|crushing_season|record_date|work_shift|ord_sample_name|        ord_id|remark|purity| brix|
+-------+---------------+-----------+----------+---------------+--------------+------+------+-----+
|    FNR|          25/26| 2026-04-13|      甲班|           甲膏|81799532788608|      | 83.26|93.66|
|    FNR|          25/26| 2026-04-13|      甲班|           甲膏|81824211188608|  null| 85.08|93.54|
|    FNR|          25/26| 2026-04-13|      甲班|           甲膏|81814164110208|  null| 85.19|93.06|
|    FNR|          25/26| 2026-04-13|      乙班|           甲膏|81856733019008|  null| 84.67|93.84|
|    FNR|          25/26| 2026-04-13|      甲班|           甲膏|81824164494208|  null| 84.18|92.94|
|    FNR|          25/26| 2026-04-12|      丙班|           甲膏|81784400526208|  null| 82.66|93.84|
|    FNR|          25/26| 2026-04-12|      丙班|           甲膏|81775372942208|  null| 82.18|95.76|
|    FNR|      

In [ ]:
# ===================== 引入必要函数 =====================
from pyspark.sql.functions import month, concat, lit, sum as spark_sum, round as spark_round, avg, count, when, col, row_number, desc
from functools import reduce
from pyspark.sql import DataFrame

# ================================================================
# ===================== 1. 计算原糖产量指标 =====================
# 原糖产量 = 入库原糖量 + 回溶糖量
# 直接基于原始数据计算，确保包含所有维度
# ================================================================

# ----- 1.1 入库原糖量（来自 lims_cpt_df 中 name='原糖'）-----
inbound_df = lims_cpt_df.filter(col("name") == "原糖") \
    .withColumn("month_num", month("rq")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .select("season", "gongsidm", "month", "banbiedm", "chanliang")

# 月度-班次
inbound_month_shift = inbound_df.groupBy("season", "gongsidm", "month", "banbiedm") \
    .agg(spark_sum("chanliang").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

# 月度-合计
inbound_month_total = inbound_df.groupBy("season", "gongsidm", "month") \
    .agg(spark_sum("chanliang").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

# 榨季累计-班次
inbound_season_shift = inbound_df.groupBy("season", "gongsidm", "banbiedm") \
    .agg(spark_sum("chanliang").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

# 榨季累计-合计
inbound_season_total = inbound_df.groupBy("season", "gongsidm") \
    .agg(spark_sum("chanliang").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

# 合并入库原糖量的四个部分
inbound_all = inbound_month_shift.unionByName(inbound_month_total) \
    .unionByName(inbound_season_shift) \
    .unionByName(inbound_season_total)

# ----- 1.2 回溶糖量（来自 lims_bbsj_df）-----
remelt_df_raw = lims_bbsj_df \
    .withColumn("month_num", month("date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .select("season", "company", "month", "banbiedm", "hrtcl") \
    .withColumnRenamed("company", "gongsidm")  # 统一公司字段名

remelt_month_shift = remelt_df_raw.groupBy("season", "gongsidm", "month", "banbiedm") \
    .agg(spark_sum("hrtcl").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

remelt_month_total = remelt_df_raw.groupBy("season", "gongsidm", "month") \
    .agg(spark_sum("hrtcl").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

remelt_season_shift = remelt_df_raw.groupBy("season", "gongsidm", "banbiedm") \
    .agg(spark_sum("hrtcl").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

remelt_season_total = remelt_df_raw.groupBy("season", "gongsidm") \
    .agg(spark_sum("hrtcl").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

remelt_all = remelt_month_shift.unionByName(remelt_month_total) \
    .unionByName(remelt_season_shift) \
    .unionByName(remelt_season_total)

# ----- 1.3 原糖产量 = 入库原糖量 + 回溶糖量 -----
# 重命名指标值列以便区分
inbound_all = inbound_all.withColumnRenamed("indicator_value", "inbound_val")
remelt_all = remelt_all.withColumnRenamed("indicator_value", "remelt_val")

# 全外连接，按榨季、公司、月份、班次
raw_sugar_result = inbound_all.join(remelt_all,
                                     on=["season", "gongsidm", "month", "banbiedm"],
                                     how="full") \
    .fillna(0) \
    .withColumn("indicator_value", spark_round(col("inbound_val") + col("remelt_val"), 2).cast("double")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value") \
    .withColumn("indicator_name", lit("原糖产量（t）")) \
    .withColumn("unit", lit("t")) \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")


# ================================================================
# ===================== 2. 计算金砂糖产量指标 =====================
# ================================================================
golden_sugar_df = lims_jst_df.select(
        col("crushing_season").alias("season"),
        col("company").alias("gongsidm"),
        col("record_date").alias("rq"),
        col("work_shift").alias("banbiedm"),
        col("golden_sugar_yield").alias("yield")
    ).filter(col("yield").isNotNull()) \
    .withColumn("month_num", month("rq")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .select("season", "gongsidm", "month", "banbiedm", "yield")

monthly_by_shift = golden_sugar_df.groupBy("season", "gongsidm", "month", "banbiedm") \
    .agg(spark_sum("yield").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = golden_sugar_df.groupBy("season", "gongsidm", "month") \
    .agg(spark_sum("yield").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = golden_sugar_df.groupBy("season", "gongsidm", "banbiedm") \
    .agg(spark_sum("yield").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = golden_sugar_df.groupBy("season", "gongsidm") \
    .agg(spark_sum("yield").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

golden_sugar_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("金砂糖产量（t）")) \
    .withColumn("unit", lit("t")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 3. 计算产糖率指标 =====================
# ================================================================
total_sugar_df = lims_cpt_df \
    .withColumn("month_num", month("rq")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .groupBy("season", "gongsidm", "month", "banbiedm") \
    .agg(spark_sum("chanliang").alias("total_sugar"))

crushing_df = lims_zzsj_df \
    .withColumn("month_num", month("tb_rq")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .groupBy("season", col("tb_gongsidm").alias("gongsidm"), "month", col("banbie_name").alias("banbiedm")) \
    .agg(spark_sum("sj_zhazheliang").alias("total_crushing"))

rate_by_shift = total_sugar_df.join(crushing_df, ["season", "gongsidm", "month", "banbiedm"], "inner") \
    .withColumn("indicator_value", spark_round(col("total_sugar") / col("total_crushing") * 100, 2).cast("double")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = total_sugar_df.groupBy("season", "gongsidm", "month") \
    .agg(spark_sum("total_sugar").alias("total_sugar")) \
    .join(
        crushing_df.groupBy("season", "gongsidm", "month").agg(spark_sum("total_crushing").alias("total_crushing")),
        ["season", "gongsidm", "month"]
    ) \
    .withColumn("indicator_value", spark_round(col("total_sugar") / col("total_crushing") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = total_sugar_df.groupBy("season", "gongsidm", "banbiedm") \
    .agg(spark_sum("total_sugar").alias("total_sugar")) \
    .join(
        crushing_df.groupBy("season", "gongsidm", "banbiedm").agg(spark_sum("total_crushing").alias("total_crushing")),
        ["season", "gongsidm", "banbiedm"]
    ) \
    .withColumn("indicator_value", spark_round(col("total_sugar") / col("total_crushing") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = total_sugar_df.groupBy("season", "gongsidm") \
    .agg(spark_sum("total_sugar").alias("total_sugar")) \
    .join(
        crushing_df.groupBy("season", "gongsidm").agg(spark_sum("total_crushing").alias("total_crushing")),
        ["season", "gongsidm"]
    ) \
    .withColumn("indicator_value", spark_round(col("total_sugar") / col("total_crushing") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

sugar_rate_result = rate_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("产糖率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 4. 计算入库原糖量指标 =====================
# ================================================================
raw_sugar_storage_df = lims_cpt_df.filter(col("name") == "原糖") \
    .withColumn("month_num", month("rq")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .select("season", "gongsidm", "month", "banbiedm", "chanliang")

monthly_by_shift = raw_sugar_storage_df.groupBy("season", "gongsidm", "month", "banbiedm") \
    .agg(spark_sum("chanliang").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = raw_sugar_storage_df.groupBy("season", "gongsidm", "month") \
    .agg(spark_sum("chanliang").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = raw_sugar_storage_df.groupBy("season", "gongsidm", "banbiedm") \
    .agg(spark_sum("chanliang").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = raw_sugar_storage_df.groupBy("season", "gongsidm") \
    .agg(spark_sum("chanliang").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

raw_sugar_storage_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("入库原糖量（t）")) \
    .withColumn("unit", lit("t")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 5. 计算回溶糖量指标 =====================
# ================================================================
remelt_sugar_df = lims_bbsj_df \
    .withColumn("month_num", month("date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .select("season", "company", "month", "banbiedm", "hrtcl")

monthly_by_shift = remelt_sugar_df.groupBy("season", "company", "month", "banbiedm") \
    .agg(spark_sum("hrtcl").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .select("season", "company", "month", "banbiedm", "indicator_value")

monthly_total = remelt_sugar_df.groupBy("season", "company", "month") \
    .agg(spark_sum("hrtcl").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .select("season", "company", "month", "banbiedm", "indicator_value")

season_by_shift = remelt_sugar_df.groupBy("season", "company", "banbiedm") \
    .agg(spark_sum("hrtcl").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "company", "month", "banbiedm", "indicator_value")

season_total = remelt_sugar_df.groupBy("season", "company") \
    .agg(spark_sum("hrtcl").alias("total")) \
    .withColumn("indicator_value", spark_round(col("total"), 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "company", "month", "banbiedm", "indicator_value")

remelt_sugar_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("回溶糖量（t）")) \
    .withColumn("unit", lit("t")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("company", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")


# ================================================================
# ===================== 6. 计算原糖色值合格率指标 =====================
# ================================================================
color_standard = lims_hgbz_df.filter(
        (col("material_name") == "原糖") & 
        (col("item_name") == "色值")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not color_standard:
    raise ValueError("未找到原糖色值合格标准！")

lower_limit = color_standard[0]["lower_limit"]
upper_limit = color_standard[0]["upper_limit"]
print(f"原糖色值合格标准: lower_limit={lower_limit}, upper_limit={upper_limit}")

raw_color_df = lims_yt_df.filter(col("raw_sugar_color_value").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("raw_sugar_color_value") >= lower_limit) & 
                     (col("raw_sugar_color_value") <= upper_limit), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = raw_color_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = raw_color_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = raw_color_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = raw_color_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

raw_color_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("原糖色值合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 7. 计算原糖水份合格率指标 =====================
# ================================================================
moisture_standard = lims_hgbz_df.filter(
        (col("material_name") == "原糖") & 
        (col("item_name") == "干燥失重")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not moisture_standard:
    raise ValueError("未找到原糖干燥失重合格标准！")

lower_limit_m = moisture_standard[0]["lower_limit"]
upper_limit_m = moisture_standard[0]["upper_limit"]
print(f"原糖干燥失重合格标准: lower_limit={lower_limit_m}, upper_limit={upper_limit_m}")

raw_moisture_df = lims_yt_df.filter(col("raw_sugar_dry_loss").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("raw_sugar_dry_loss") >= lower_limit_m) & 
                     (col("raw_sugar_dry_loss") <= upper_limit_m), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = raw_moisture_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = raw_moisture_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = raw_moisture_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = raw_moisture_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

raw_moisture_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("原糖水份合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 8. 计算回溶糖浆锤度合格率指标 =====================
# ================================================================
remelt_brix_standard = lims_hgbz_df.filter(
        (col("material_name") == "洄溶糖浆") & 
        (col("item_name") == "锤度")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not remelt_brix_standard:
    raise ValueError("未找到洄溶糖浆锤度合格标准！")

lower_limit_remelt_brix = remelt_brix_standard[0]["lower_limit"]
upper_limit_remelt_brix = remelt_brix_standard[0]["upper_limit"]
print(f"洄溶糖浆锤度合格标准: lower_limit={lower_limit_remelt_brix}, upper_limit={upper_limit_remelt_brix}")

remelt_brix_df = lims_hrtj_df.filter(col("remelt_syrup_brix").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("remelt_syrup_brix") >= lower_limit_remelt_brix) & 
                     (col("remelt_syrup_brix") <= upper_limit_remelt_brix), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = remelt_brix_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = remelt_brix_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = remelt_brix_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = remelt_brix_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

remelt_brix_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("回溶糖浆锤度合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 9. 计算回溶糖浆色值合格率指标 =====================
# ================================================================
remelt_color_standard = lims_hgbz_df.filter(
        (col("material_name") == "洄溶糖浆") & 
        (col("item_name") == "色值")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not remelt_color_standard:
    raise ValueError("未找到洄溶糖浆色值合格标准！")

lower_limit_remelt_color = remelt_color_standard[0]["lower_limit"]
upper_limit_remelt_color = remelt_color_standard[0]["upper_limit"]
print(f"洄溶糖浆色值合格标准: lower_limit={lower_limit_remelt_color}, upper_limit={upper_limit_remelt_color}")

remelt_color_df = lims_hrtj_df.filter(col("remelt_syrup_color_value").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("remelt_syrup_color_value") >= lower_limit_remelt_color) & 
                     (col("remelt_syrup_color_value") <= upper_limit_remelt_color), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = remelt_color_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = remelt_color_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = remelt_color_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = remelt_color_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

remelt_color_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("回溶糖浆色值合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 10. 计算金砂糖糖膏纯度合格率指标 =====================
# ================================================================
lower_limit_paste = 70.0
upper_limit_paste = 78.0
print(f"金砂糖糖膏纯度合格标准（固定）: lower_limit={lower_limit_paste}, upper_limit={upper_limit_paste}")

golden_paste_purity_df = lims_jstg_df \
    .filter(col("remark") == "金砂糖") \
    .filter(col("purity").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("purity") >= lower_limit_paste) & 
                     (col("purity") <= upper_limit_paste), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = golden_paste_purity_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = golden_paste_purity_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = golden_paste_purity_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = golden_paste_purity_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

golden_paste_purity_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("金砂糖糖膏纯度合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 11. 计算金砂糖糖膏锤度合格率指标 =====================
# ================================================================
lower_limit_brix = 94.0
upper_limit_brix = 99.0
print(f"金砂糖糖膏锤度合格标准（固定）: lower_limit={lower_limit_brix}, upper_limit={upper_limit_brix}")

golden_paste_brix_df = lims_jstg_df \
    .filter(col("remark") == "金砂糖") \
    .filter(col("brix").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("brix") >= lower_limit_brix) & 
                     (col("brix") <= upper_limit_brix), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = golden_paste_brix_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = golden_paste_brix_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = golden_paste_brix_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = golden_paste_brix_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

golden_paste_brix_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("金砂糖糖膏锤度合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 12. 计算金砂糖色值合格率指标 =====================
# ================================================================
golden_color_standard = lims_hgbz_df.filter(
        (col("material_name") == "金砂糖") & 
        (col("item_name") == "色值")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not golden_color_standard:
    raise ValueError("未找到金砂糖色值合格标准！")

lower_limit_golden_color = golden_color_standard[0]["lower_limit"]
upper_limit_golden_color = golden_color_standard[0]["upper_limit"]
print(f"金砂糖色值合格标准: lower_limit={lower_limit_golden_color}, upper_limit={upper_limit_golden_color}")

golden_color_df = lims_jst_df.filter(col("golden_sugar_color_value").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("golden_sugar_color_value") >= lower_limit_golden_color) & 
                     (col("golden_sugar_color_value") <= upper_limit_golden_color), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = golden_color_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = golden_color_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = golden_color_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = golden_color_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

golden_color_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("金砂糖色值合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 13. 计算金砂糖水份合格率指标 =====================
# ================================================================
golden_moisture_standard = lims_hgbz_df.filter(
        (col("material_name") == "金砂糖") & 
        (col("item_name") == "干燥失重")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not golden_moisture_standard:
    raise ValueError("未找到金砂糖干燥失重合格标准！")

lower_limit_golden_moisture = golden_moisture_standard[0]["lower_limit"]
upper_limit_golden_moisture = golden_moisture_standard[0]["upper_limit"]
print(f"金砂糖干燥失重合格标准: lower_limit={lower_limit_golden_moisture}, upper_limit={upper_limit_golden_moisture}")

golden_moisture_df = lims_jst_df.filter(col("golden_sugar_dry_loss").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("golden_sugar_dry_loss") >= lower_limit_golden_moisture) & 
                     (col("golden_sugar_dry_loss") <= upper_limit_golden_moisture), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = golden_moisture_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = golden_moisture_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = golden_moisture_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = golden_moisture_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

golden_moisture_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("金砂糖水份合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 14. 计算金砂糖糖度合格率指标 =====================
# ================================================================
golden_brix_standard = lims_hgbz_df.filter(
        (col("material_name") == "金砂糖") & 
        (col("item_name") == "糖度")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not golden_brix_standard:
    raise ValueError("未找到金砂糖糖度合格标准！")

lower_limit_golden_brix = golden_brix_standard[0]["lower_limit"]
upper_limit_golden_brix = golden_brix_standard[0]["upper_limit"]
print(f"金砂糖糖度合格标准: lower_limit={lower_limit_golden_brix}, upper_limit={upper_limit_golden_brix}")

golden_brix_df = lims_jst_df.filter(col("golden_sugar_brix").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("golden_sugar_brix") >= lower_limit_golden_brix) & 
                     (col("golden_sugar_brix") <= upper_limit_golden_brix), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = golden_brix_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = golden_brix_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = golden_brix_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = golden_brix_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

golden_brix_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("金砂糖糖度合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 15. 计算桔水产率指标 =====================
# ================================================================
cane_molasses_yield_df = lims_zzjs_df.filter(col("cane_molasses_yield").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("cane_molasses_yield").alias("total_molasses"))

crushing_df2 = lims_zzsj_df \
    .withColumn("month_num", month("tb_rq")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .groupBy("season", col("tb_gongsidm").alias("company"), "month", col("banbie_name").alias("work_shift")) \
    .agg(spark_sum("sj_zhazheliang").alias("total_crushing"))

rate_by_shift2 = cane_molasses_yield_df.join(
        crushing_df2,
        (cane_molasses_yield_df.crushing_season == crushing_df2.season) &
        (cane_molasses_yield_df.company == crushing_df2.company) &
        (cane_molasses_yield_df.month == crushing_df2.month) &
        (cane_molasses_yield_df.work_shift == crushing_df2.work_shift),
        "inner"
    ) \
    .select(cane_molasses_yield_df.crushing_season.alias("season"),
            cane_molasses_yield_df.company.alias("gongsidm"),
            cane_molasses_yield_df.month,
            cane_molasses_yield_df.work_shift.alias("banbiedm"),
            col("total_molasses"),
            col("total_crushing")) \
    .withColumn("indicator_value", spark_round(col("total_molasses") / col("total_crushing") * 100, 2).cast("double")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total2 = cane_molasses_yield_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("total_molasses").alias("total_molasses")) \
    .join(
        crushing_df2.groupBy("season", "company", "month").agg(spark_sum("total_crushing").alias("total_crushing")),
        (cane_molasses_yield_df.crushing_season == crushing_df2.season) &
        (cane_molasses_yield_df.company == crushing_df2.company) &
        (cane_molasses_yield_df.month == crushing_df2.month),
        "inner"
    ) \
    .select(cane_molasses_yield_df.crushing_season.alias("season"),
            cane_molasses_yield_df.company.alias("gongsidm"),
            cane_molasses_yield_df.month,
            col("total_molasses"),
            col("total_crushing")) \
    .withColumn("indicator_value", spark_round(col("total_molasses") / col("total_crushing") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift2 = cane_molasses_yield_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("total_molasses").alias("total_molasses")) \
    .join(
        crushing_df2.groupBy("season", "company", "work_shift").agg(spark_sum("total_crushing").alias("total_crushing")),
        (cane_molasses_yield_df.crushing_season == crushing_df2.season) &
        (cane_molasses_yield_df.company == crushing_df2.company) &
        (cane_molasses_yield_df.work_shift == crushing_df2.work_shift),
        "inner"
    ) \
    .select(cane_molasses_yield_df.crushing_season.alias("season"),
            cane_molasses_yield_df.company.alias("gongsidm"),
            cane_molasses_yield_df.work_shift.alias("banbiedm"),
            col("total_molasses"),
            col("total_crushing")) \
    .withColumn("indicator_value", spark_round(col("total_molasses") / col("total_crushing") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total2 = cane_molasses_yield_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("total_molasses").alias("total_molasses")) \
    .join(
        crushing_df2.groupBy("season", "company").agg(spark_sum("total_crushing").alias("total_crushing")),
        (cane_molasses_yield_df.crushing_season == crushing_df2.season) &
        (cane_molasses_yield_df.company == crushing_df2.company),
        "inner"
    ) \
    .select(cane_molasses_yield_df.crushing_season.alias("season"),
            cane_molasses_yield_df.company.alias("gongsidm"),
            col("total_molasses"),
            col("total_crushing")) \
    .withColumn("indicator_value", spark_round(col("total_molasses") / col("total_crushing") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

molasses_rate_result = rate_by_shift2.unionByName(monthly_total2) \
    .unionByName(season_by_shift2) \
    .unionByName(season_total2) \
    .withColumn("indicator_name", lit("桔水产率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 16. 计算丙膏平均纯度指标 =====================
# ================================================================
propyl_purity_df = lims_bg_df.filter(col("propyl_paste_apparent_purity").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .select("crushing_season", "company", "month", "work_shift", "propyl_paste_apparent_purity")

monthly_by_shift = propyl_purity_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_round(avg("propyl_paste_apparent_purity"), 2).alias("avg_purity")) \
    .withColumn("indicator_value", col("avg_purity").cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = propyl_purity_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_round(avg("propyl_paste_apparent_purity"), 2).alias("avg_purity")) \
    .withColumn("indicator_value", col("avg_purity").cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = propyl_purity_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_round(avg("propyl_paste_apparent_purity"), 2).alias("avg_purity")) \
    .withColumn("indicator_value", col("avg_purity").cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = propyl_purity_df.groupBy("crushing_season", "company") \
    .agg(spark_round(avg("propyl_paste_apparent_purity"), 2).alias("avg_purity")) \
    .withColumn("indicator_value", col("avg_purity").cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

propyl_purity_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("丙膏平均纯度（AP）")) \
    .withColumn("unit", lit("/")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 17. 计算桔水平均纯度指标 =====================
# ================================================================
cane_purity_df = lims_zzjs_df.filter(col("cane_molasses_apparent_purity").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .select("crushing_season", "company", "month", "work_shift", "cane_molasses_apparent_purity")

monthly_by_shift = cane_purity_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_round(avg("cane_molasses_apparent_purity"), 2).alias("avg_purity")) \
    .withColumn("indicator_value", col("avg_purity").cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = cane_purity_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_round(avg("cane_molasses_apparent_purity"), 2).alias("avg_purity")) \
    .withColumn("indicator_value", col("avg_purity").cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = cane_purity_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_round(avg("cane_molasses_apparent_purity"), 2).alias("avg_purity")) \
    .withColumn("indicator_value", col("avg_purity").cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = cane_purity_df.groupBy("crushing_season", "company") \
    .agg(spark_round(avg("cane_molasses_apparent_purity"), 2).alias("avg_purity")) \
    .withColumn("indicator_value", col("avg_purity").cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

cane_purity_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("桔水平均纯度（AP）")) \
    .withColumn("unit", lit("/")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 18. 计算丙膏桔水纯度差指标 =====================
# ================================================================
propyl_purity_for_diff = propyl_purity_result.select(
    col("season"), col("company"), col("shift"), col("month_period"),
    col("indicator_value").alias("propyl_purity")
)
cane_purity_for_diff = cane_purity_result.select(
    col("season"), col("company"), col("shift"), col("month_period"),
    col("indicator_value").alias("cane_purity")
)
purity_diff_result = propyl_purity_for_diff.join(cane_purity_for_diff, ["season", "company", "shift", "month_period"], "inner") \
    .withColumn("indicator_value", spark_round(col("propyl_purity") - col("cane_purity"), 2).cast("double")) \
    .select("season", "company", "shift", "month_period",
            lit("丙膏桔水纯度差（AP）").alias("indicator_name"),
            col("indicator_value"),
            lit("/").alias("unit"))

# ================================================================
# ===================== 19. 计算丙糖糊色值合格率指标 =====================
# ================================================================
b_paste_color_standard_sql = """
SELECT 
    minimum_value,
    maximum_value 
FROM dwd.dwd_mes_process_standard_f_1h 
WHERE project_name = 'B糖糊色值' 
ORDER BY create_time DESC 
LIMIT 1
"""
b_paste_color_standard_df = spark.sql(b_paste_color_standard_sql)
if b_paste_color_standard_df.count() == 0:
    raise ValueError("未找到 B糖糊色值 的合格标准！")

standard_row = b_paste_color_standard_df.collect()[0]
lower_limit_third_color = standard_row["minimum_value"]
upper_limit_third_color = standard_row["maximum_value"]
print(f"丙糖糊色值合格标准（来自MES）: lower_limit={lower_limit_third_color}, upper_limit={upper_limit_third_color}")

third_paste_color_df = lims_bth_df.filter(col("third_sugar_paste_color_value").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("third_sugar_paste_color_value") >= lower_limit_third_color) & 
                     (col("third_sugar_paste_color_value") <= upper_limit_third_color), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = third_paste_color_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = third_paste_color_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = third_paste_color_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = third_paste_color_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

third_paste_color_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("丙糖糊色值合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 20. 计算丙糖糊锤度合格率指标 =====================
# ================================================================
third_paste_brix_standard = lims_hgbz_df.filter(
        (col("material_name") == "丙糖糊") & 
        (col("item_name") == "锤度")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not third_paste_brix_standard:
    raise ValueError("未找到丙糖糊锤度合格标准！")

lower_limit_third_brix = third_paste_brix_standard[0]["lower_limit"]
upper_limit_third_brix = third_paste_brix_standard[0]["upper_limit"]
print(f"丙糖糊锤度合格标准: lower_limit={lower_limit_third_brix}, upper_limit={upper_limit_third_brix}")

third_paste_brix_df = lims_bth_df.filter(col("third_sugar_paste_brix").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("third_sugar_paste_brix") >= lower_limit_third_brix) & 
                     (col("third_sugar_paste_brix") <= upper_limit_third_brix), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = third_paste_brix_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = third_paste_brix_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = third_paste_brix_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = third_paste_brix_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

third_paste_brix_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("丙糖糊锤度合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 21. 计算桔水重力纯度合格率指标 =====================
# ================================================================
cane_gravity_standard = lims_hgbz_df.filter(
        (col("material_name") == "榨蔗桔水") & 
        (col("item_name") == "重力纯度")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not cane_gravity_standard:
    raise ValueError("未找到榨蔗桔水重力纯度合格标准！")

lower_limit_gravity = cane_gravity_standard[0]["lower_limit"]
upper_limit_gravity = cane_gravity_standard[0]["upper_limit"]
print(f"榨蔗桔水重力纯度合格标准: lower_limit={lower_limit_gravity}, upper_limit={upper_limit_gravity}")

cane_gravity_df = lims_zzjs_df.filter(col("cane_molasses_gravity_purity").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("cane_molasses_gravity_purity") >= lower_limit_gravity) & 
                     (col("cane_molasses_gravity_purity") <= upper_limit_gravity), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = cane_gravity_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = cane_gravity_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = cane_gravity_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = cane_gravity_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

cane_gravity_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("桔水重力纯度合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 22. 计算桔水锤度合格率指标 =====================
# ================================================================
cane_brix_standard = lims_hgbz_df.filter(
        (col("material_name") == "榨蔗桔水") & 
        (col("item_name") == "锤度")
    ) \
    .orderBy(col("dismonth_date").desc()) \
    .limit(1) \
    .select("lower_limit", "upper_limit") \
    .collect()

if not cane_brix_standard:
    raise ValueError("未找到榨蔗桔水锤度合格标准！")

lower_limit_brix_cane = cane_brix_standard[0]["lower_limit"]
upper_limit_brix_cane = cane_brix_standard[0]["upper_limit"]
print(f"榨蔗桔水锤度合格标准: lower_limit={lower_limit_brix_cane}, upper_limit={upper_limit_brix_cane}")

cane_brix_df = lims_zzjs_df.filter(col("cane_molasses_brix").isNotNull()) \
    .withColumn("month_num", month("record_date")) \
    .withColumn("month", concat(col("month_num"), lit("月"))) \
    .withColumn("is_qualified", 
                when((col("cane_molasses_brix") >= lower_limit_brix_cane) & 
                     (col("cane_molasses_brix") <= upper_limit_brix_cane), 1)
                .otherwise(0)) \
    .select("crushing_season", "company", "month", "work_shift", "is_qualified")

monthly_by_shift = cane_brix_df.groupBy("crushing_season", "company", "month", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

monthly_total = cane_brix_df.groupBy("crushing_season", "company", "month") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_by_shift = cane_brix_df.groupBy("crushing_season", "company", "work_shift") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .withColumnRenamed("work_shift", "banbiedm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

season_total = cane_brix_df.groupBy("crushing_season", "company") \
    .agg(spark_sum("is_qualified").alias("qualified_count"), count("*").alias("total_count")) \
    .withColumn("indicator_value", spark_round(col("qualified_count") / col("total_count") * 100, 2).cast("double")) \
    .withColumn("banbiedm", lit("工段合计")) \
    .withColumn("month", lit("榨季累计")) \
    .withColumnRenamed("crushing_season", "season") \
    .withColumnRenamed("company", "gongsidm") \
    .select("season", "gongsidm", "month", "banbiedm", "indicator_value")

cane_brix_pass_rate_result = monthly_by_shift.unionByName(monthly_total) \
    .unionByName(season_by_shift) \
    .unionByName(season_total) \
    .withColumn("indicator_name", lit("桔水锤度合格率（%）")) \
    .withColumn("unit", lit("%")) \
    .withColumnRenamed("season", "season") \
    .withColumnRenamed("gongsidm", "company") \
    .withColumnRenamed("month", "month_period") \
    .withColumnRenamed("banbiedm", "shift") \
    .select("season", "company", "shift", "month_period", "indicator_name", "indicator_value", "unit")

# ================================================================
# ===================== 合并所有指标 =====================
# ================================================================
result_dfs = [
    raw_sugar_result,
    golden_sugar_result,
    sugar_rate_result,
    raw_sugar_storage_result,
    remelt_sugar_result,
    raw_color_pass_rate_result,
    raw_moisture_pass_rate_result,
    remelt_brix_pass_rate_result,
    remelt_color_pass_rate_result,
    golden_paste_purity_pass_rate_result,
    golden_paste_brix_pass_rate_result,
    golden_color_pass_rate_result,
    golden_moisture_pass_rate_result,
    golden_brix_pass_rate_result,
    molasses_rate_result,
    propyl_purity_result,
    cane_purity_result,
    purity_diff_result,
    third_paste_color_pass_rate_result,
    third_paste_brix_pass_rate_result,
    cane_gravity_pass_rate_result,
    cane_brix_pass_rate_result
]

def ensure_double_type(df):
    if "indicator_value" in df.columns:
        return df.withColumn("indicator_value", col("indicator_value").cast("double"))
    return df

result_dfs = [ensure_double_type(df) for df in result_dfs]

final_report_df = reduce(lambda a, b: a.unionByName(b), result_dfs)

# 最终展示
# final_report_df.orderBy("season", "company", "indicator_name", "month_period", "shift") \
#     .show(200, truncate=False)

# 写入 Hive 表
target_table = "app.app_lims_season_centrifugal_indicator_stats_f_1h"

final_report_df.write.mode("overwrite").saveAsTable(target_table)
spark.stop()
print("已写入，spark关闭")